In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_7 import BioJepa, BioJepaConfig
from training_v0_7 import create_model, maybe_compile
from config_v0_7 import DataConfig
from evals.evals import EvalContext, run_encoder_evals, run_composer_evals, run_ac_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/jepa/v0_7').expanduser()
ref_root = Path('~/data/jepa/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir=ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cpu


In [3]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=2, #6
    heads=2, #4
    embed_dim=8, #256
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=2.38,
    film_linear_multiple=0.81,
    sim_coeff=25,
    std_coeff=25,
    cov_coeff=1,
    pert_latent_dim=128,
    pert_mode_dim=64,
)

EVAL_BATCH_SIZE = 32

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters()):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters()):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters()):,}')

Student/Teacher: 84,086
ACpredictor: 85,824
PerturbationComposer: 510,400


### Load Model 

In [5]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_ac_final.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Encoder Training Evals

In [ ]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
pt_eval_results = run_encoder_evals(eval_ctx)

save_report(pt_eval_results, data_cfg.eval_results_dir / 'encoder_eval_report.json')
pt_eval_results

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Alignment Training Eval

In [ ]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
align_eval_results = run_composer_evals(align_eval_ctx)

save_report(align_eval_results, data_cfg.eval_results_dir / 'composer_eval_report.json')
align_eval_results

In [ ]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()

### ACPredictor Eval

In [6]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_v0_7_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
keys = decoder.load_state_dict(decoder_sd)
keys

<All keys matched successfully>

In [ ]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED,
})
full_eval_results = run_ac_evals(eval_ctx)

save_report(full_eval_results, data_cfg.eval_results_dir / 'ac_eval_report.json')
full_eval_results

Using cpu
Loaded cached test inference from /Users/djemec/data/jepa/v0_7/test_inference_cache (135 shards, delete directory to recompute)
expression_prediction: Pearson=0.9832, R2=0.9453, Centroid_acc=0.0015
gene_level_analysis: Dir_acc=0.9851, Top50_acc=0.6715
Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


perturbation_retrieval (dna): 100%|█████████████████████████████████| 5/5 [19:56<00:00, 239.36s/it]


perturbation_retrieval (dna): MRR=0.0004


perturbation_retrieval (chemical): 100%|█████████████████████████████| 5/5 [00:20<00:00,  4.15s/it]


perturbation_retrieval (chemical): MRR=0.0207


In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()